# TRELLIS image -> 3D building (Colab GPU) — tách rời 2 bước

Bài học: trộn deps của repo vào môi trường TRELLIS gây xung đột liên hoàn (Pillow `_Ink`,
NumPy `_center`...). Vì vậy notebook này **chỉ chạy TRELLIS trên GPU để xuất `building.glb`**
(KHÔNG import `builder`, KHÔNG cài thêm gì). Việc lắp `SceneBundle` (describer tiếng Việt,
thuần CPU) làm ở **máy local sạch** bằng `python -m builder.run --from-glb building.glb` (Bước 7).

**Yêu cầu:** Runtime = GPU (Runtime -> Change runtime type -> T4/L4/A100, >=6GB VRAM).
Nếu môi trường đã bẩn (đã lỡ cài đè), dùng Runtime -> Disconnect and delete runtime để làm lại sạch.

In [ ]:
!nvidia-smi -L  # xác nhận có GPU

## Bước 1 - Clone TRELLIS (chỉ TRELLIS, không cần repo trên GPU)

In [ ]:
%cd /content
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS

## Bước 2 - Cài TRELLIS theo README của họ
KHÔNG cài thêm trimesh/pydantic/pillow ở đây — để TRELLIS tự quản môi trường (tránh xung đột).

In [ ]:
%cd /content/TRELLIS
!. ./setup.sh --basic --xformers --spconv --mipgaussian --nvdiffrast
%cd /content

## Bước 3 - Tải lên ảnh tòa nhà

In [ ]:
from google.colab import files
up = files.upload()
image_path = '/content/' + list(up)[0]
print('image:', image_path)

## Bước 4 - Chạy TRELLIS -> xuất `building.glb`
Dùng trực tiếp API TRELLIS (không đụng tới `builder`).

In [ ]:
import os, sys
os.environ.setdefault('ATTN_BACKEND', 'xformers')
os.environ.setdefault('SPCONV_ALGO', 'native')
sys.path.insert(0, '/content/TRELLIS')

from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipe.cuda()

img = Image.open(image_path).convert('RGB')
out = pipe.run(img, seed=1)
glb = postprocessing_utils.to_glb(out['gaussian'][0], out['mesh'][0], simplify=0.95, texture_size=1024)
glb.export('building.glb')
print('saved /content/building.glb')

## Bước 5 - Xem GLB inline

In [ ]:
import base64
from IPython.display import HTML
b64 = base64.b64encode(open('building.glb', 'rb').read()).decode()
HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}" camera-controls auto-rotate
  shadow-intensity="0.7" exposure="1.0" style="width:100%;height:460px;background:#f0eee6;border-radius:16px"></model-viewer>
''')

## Bước 6 - Tải `building.glb` về

In [ ]:
from google.colab import files
files.download('building.glb')

## Bước 7 - (Máy CPU sạch) lắp SceneBundle quanh GLB
Trên máy của bạn (đã `pip install -r requirements.txt`), chạy:

```bash
python -m builder.run --from-glb building.glb \
  --name "Tòa nhà từ ảnh" --space office \
  --prompt "Mô tả ngắn cho tòa nhà trong ảnh" \
  --audience "Khách hàng mục tiêu" \
  --out frontend/public/artifacts
```

Tạo `<id>.glb` + `<id>.json` (backend=generative, không có drill phòng) để demo đọc.
Hoàn toàn CPU, không đụng môi trường TRELLIS.

Nếu TRELLIS setup cứ lỗi trên Colab, đổi sang **Hunyuan3D** (cài nhẹ hơn) để xuất `building.glb`,
rồi vẫn dùng đúng Bước 7 ở trên.